In [1]:
import shutil
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torchvision
import kagglehub
from math import floor

/Users/ilya/Documents/CV_train_DL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_ROOT = Path('/Users/ilya/Documents/CV_train_DL/data')
BATCHES_DIR = DATA_ROOT / 'cifar-10-batches-py'


def ensure_cifar10_from_kaggle(data_root: Path = DATA_ROOT) -> Path:
    """Download CIFAR-10 from Kaggle and place batches under data_root."""
    data_root.mkdir(parents=True, exist_ok=True)
    batches_dir = data_root / 'cifar-10-batches-py'

    if (batches_dir / 'data_batch_1').exists() and (batches_dir / 'test_batch').exists():
        return batches_dir

    # Drop incomplete torchvision download if present.
    broken_archive = data_root / 'cifar-10-python.tar.gz'
    if broken_archive.exists() and broken_archive.stat().st_size < 170_000_000:
        broken_archive.unlink()

    cache_path = Path(
        kagglehub.dataset_download(
            'pankrzysiu/cifar10-python',
            output_dir=str(data_root / 'kaggle_cifar10'),
        )
    )

    # Dataset may contain extracted batches and/or the .tar.gz archive.
    candidates = [
        cache_path / 'cifar-10-batches-py',
        cache_path / 'cifar10-python' / 'cifar-10-batches-py',
    ]
    for candidate in candidates:
        if (candidate / 'data_batch_1').exists():
            if batches_dir.exists():
                shutil.rmtree(batches_dir)
            shutil.copytree(candidate, batches_dir)
            return batches_dir

    archives = list(cache_path.rglob('cifar-10-python.tar.gz'))
    if not archives:
        raise FileNotFoundError(f'CIFAR-10 archive not found under {cache_path}')

    with tarfile.open(archives[0], 'r:gz') as tar:
        tar.extractall(path=data_root)

    if not (batches_dir / 'data_batch_1').exists():
        raise FileNotFoundError(f'Failed to extract CIFAR-10 batches into {batches_dir}')
    return batches_dir


ensure_cifar10_from_kaggle()

train_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

train_set = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=True,
                             download=False, transform=train_transform)
test_set = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=False,
                            download=False, transform=test_transform)

print(len(train_set), len(test_set), train_set.classes)



50000 10000 ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
def out_count(tensor, kernel, padding, stride):
    tensor = np.pad(tensor, ((0, 0), (padding, padding), (padding, padding)), mode='constant', constant_values=0)
    i_size = floor((tensor.shape[1] - kernel.shape[2]) / stride) + 1
    j_size = floor((tensor.shape[2] - kernel.shape[3]) / stride) + 1
    c_out = kernel.shape[0]
    out_sum = np.zeros((c_out, i_size, j_size))
    for c in range(c_out):
        for i in range(i_size):
            for j in range(j_size):
                    out_sum[c][i][j] = (tensor[:, i*stride:(kernel.shape[2]+i*stride), j*stride:(kernel.shape[3]+j*stride)]*kernel[c]).sum()
    return out_sum

In [4]:
def grad_kernel(dout, X, kh, kw, padding):
    C_out = dout.shape[0]
    C_in = X.shape[0]
    X = np.pad(X, ((0, 0), (padding, padding), (padding, padding)), mode='constant', constant_values=0)
    dK = np.zeros((C_out, C_in, kh, kw))
    for oc in range(C_out):
        for ic in range(C_in):        
            for m in range(kh):
                for n in range(kw):
                    dK[oc][ic][m][n] = (dout[oc] *  X[ic, m:m+dout.shape[1], n:n+dout.shape[2]]).sum()
    return dK

In [5]:
def grad_input(dout, K, padding, stride):
    P_rev = K.shape[2] - 1 - padding  
    K_flipped = K[:, :, ::-1, ::-1]
    C_out = dout.shape[0]
    C_in = K.shape[1]
    dX = np.zeros((C_in, (dout.shape[1]-1)*stride+K_flipped.shape[2]-2*padding, (dout.shape[2]-1)*stride+K_flipped.shape[3]-2*padding))
    for ic in range(C_in):  
        for oc in range(C_out):
            dX[ic] += out_count(dout[oc][np.newaxis], K_flipped[oc, ic][np.newaxis, np.newaxis], P_rev, 1)[0]
    return dX

In [6]:
def grad_bias(dout):
    return dout.sum(axis=(1, 2))

In [7]:
class Conv2D :
    def __init__(self, C_in, C_out, kh, kw, padding=0, stride=1):
        std = np.sqrt(2.0 / (C_in * kh * kw))
        self.K = np.random.randn(C_out, C_in, kh, kw) * std
        self.b = np.zeros(C_out)
        self.stride = stride
        self.padding = padding
        self.cache = None
        self.kh = kh
        self.kw = kw
    
    def forward(self, X):
        self.cache = X
        N = X.shape[0]
        outs = []
        for n in range(N):
            outs.append(out_count(X[n], self.K, self.padding, self.stride))
        out = np.stack(outs) + self.b[None, :, None, None]
        return out
    
    def update(self, lr):
        self.K -= lr * self.dK
        self.b -= lr * self.db
    

    def backward(self, dout):
        X = self.cache
        N = X.shape[0]
        self.dK = np.zeros_like(self.K)
        self.db = np.zeros_like(self.b)
        dX = np.zeros_like(X)
        for n in range(N):
            self.dK += grad_kernel(dout[n], X[n], self.kh, self.kw, self.padding)
            self.db += grad_bias(dout[n])
            dX[n] = grad_input(dout[n], self.K, self.padding, self.stride)
        return dX






In [8]:
class MaxPool:
    def __init__(self, size=2, padding=0, stride=2):
        self.padding = padding
        self.stride = stride
        self.mask = None
        self.size = size

    def forward(self, X):
        N, C, H, W = X.shape
        H_out = (H - self.size) // self.stride + 1
        W_out = (W - self.size) // self.stride + 1
        out = np.zeros((N, C, H_out, W_out))
        mask = np.zeros_like(X)
        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        hs, ws = i * self.stride, j * self.stride
                        window = X[n, c, hs:hs+self.size, ws:ws+self.size]
                        mi, mj = np.unravel_index(np.argmax(window), window.shape)
                        out[n, c, i, j] = window[mi, mj]
                        mask[n, c, hs + mi, ws + mj] = 1
        self.mask = mask
        return out

    def backward(self, dout):
        dout_up = np.repeat(np.repeat(dout, self.size, axis=2), self.size, axis=3)
        return dout_up * self.mask

In [9]:
class Flatten:
    def __init__(self):
        self.cache = None
    
    def forward(self, X):
        self.cache = X.shape
        return X.reshape(X.shape[0], -1)
    
    def backward(self, dout):
        return dout.reshape(self.cache)



In [10]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * 0.01
        self.b = np.zeros(out_features)
        self.cache = None

    def forward(self, X):
        self.cache = X
        return X @ self.W + self.b

    def update(self, lr):
        self.W -= lr * self.dW
        self.b -= lr * self.db

    def backward(self, dout):
        X = self.cache
        self.dW = X.T @ dout
        self.db = dout.sum(axis=0)
        dX = dout @ self.W.T
        return dX



In [11]:
class Sequential:
    def __init__(self, *layers):
        self.layers = list(layers)

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

In [12]:
class CrossEntropyLoss:
    def forward(self, logits, y):
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(shifted)
        self.softmax = exp / exp.sum(axis=1, keepdims=True)
        self.y = y
        N = logits.shape[0]
        correct_probs = self.softmax[np.arange(N), y]
        loss = -np.mean(np.log(correct_probs + 1e-12))
        return loss

    def backward(self):
        N = self.softmax.shape[0]
        dout = self.softmax.copy()
        dout[np.arange(N), self.y] -= 1
        return dout / N

In [13]:
class RELU:
    def __init__(self):
        self.cache = None

    def forward(self, x):
        self.cache = x
        out = np.maximum(0, x)
        return out
    
    def backward(self, dout):
        x = self.cache
        dx = (x > 0).astype(float) * dout
        return dx

In [14]:
def sgd_step(layer, lr):
    layer.K -= lr*layer.dK
    layer.b -= lr*layer.db

In [15]:
model = Sequential(
    Conv2D(3, 8, 3, 3, padding=1),
    RELU(),
    MaxPool(size=2, stride=2),
    Conv2D(8, 16, 3, 3, padding=1),
    RELU(),
    MaxPool(size=2, stride=2),
    Flatten(),
    Linear(1024, 10),
)


In [16]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)

criterion = CrossEntropyLoss()

epochs = 10
lr = 0.05
batch_size = 32
N = 500
X_list = []
y_list = []
for i in range(N):
    img, label = train_set[i]
    X_list.append(img.numpy())
    y_list.append(label)

X_train = np.stack(X_list)
y_train = np.array(y_list)  

for epoch in range(epochs):
    indices = np.random.permutation(N)
    epoch_loss = 0.0
    num_batches = 0

    for start in range(0, N, batch_size):
        idx = indices[start:start + batch_size]
        X_batch = X_train[idx]
        y_batch = y_train[idx]

        # forward
        logits = model.forward(X_batch)
        loss = criterion.forward(logits, y_batch)

        # backward
        dout = criterion.backward()
        model.backward(dout)

        # update
        for layer in model.layers:
            if hasattr(layer, 'update'):
                layer.update(lr)

        epoch_loss += loss
        num_batches += 1

    print(f"Epoch {epoch}: loss={epoch_loss / num_batches:.4f}")

Epoch 0: loss=2.7002
Epoch 1: loss=2.1145
Epoch 2: loss=1.9182
Epoch 3: loss=1.6805
Epoch 4: loss=1.5184
Epoch 5: loss=1.4180
Epoch 6: loss=1.2242
Epoch 7: loss=1.1592
Epoch 8: loss=0.9471
Epoch 9: loss=0.8769
